# FF++ C23 external visual evaluation

This notebook evaluates the five frozen SDFVD 2.0 paired-source CV checkpoints on the FaceForensics++ C23 subtree from the approved Kaggle collection. It does **not** train or tune on FF++. The evaluation uses the SDFVD2 validation-selected visual threshold for each fold and averages five evenly spaced frame scores into one score per video sequence.

The Kaggle listing is a mixed collection. The download stays in Colab temporary storage; only evaluation reports are written to Drive. The notebook extracts only the FF++ subtree. Before running, set `BASE` to the Drive folder containing your checkpoints and source bundle, and set `CV_RUN_FOLDER` to your paired-source CV run folder. Add Kaggle credentials to Colab Secrets as `KAGGLE_USERNAME` and `KAGGLE_KEY`; never paste credentials into a cell or save them to Drive.


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import base64, json, os, shutil, subprocess, sys, time, zipfile

drive.mount('/content/drive')
# Change this one path in Colab to the Drive folder containing the dataset artifacts.
BASE = Path('/content/drive/MyDrive/Datasets')
CV_RUN_FOLDER = 'REPLACE_WITH_YOUR_SDFVD2_CV_RUN_FOLDER'
CV_ROOT = BASE / CV_RUN_FOLDER / 'group_cv'
SOURCE_B64 = BASE / 'poor_guys_deepfake_colab_source.b64'
DEST_ROOT = BASE / 'kaggle_ffpp_c23_external_validation'
DEST_ROOT.mkdir(parents=True, exist_ok=True)
assert CV_ROOT.is_dir(), f'Missing paired-source checkpoints: {CV_ROOT}'
assert SOURCE_B64.is_file(), f'Missing project source bundle: {SOURCE_B64}'

username = userdata.get('KAGGLE_USERNAME')
api_key = userdata.get('KAGGLE_KEY')
if not username or not api_key:
    raise RuntimeError('Add KAGGLE_USERNAME and KAGGLE_KEY in Colab Secrets, enable notebook access, then rerun this cell.')
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)
(kaggle_dir / 'kaggle.json').write_text(json.dumps({'username': username, 'key': api_key}), encoding='utf-8')
os.chmod(kaggle_dir / 'kaggle.json', 0o600)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)

DOWNLOAD = Path('/content/kaggle_ffpp_download')
DOWNLOAD.mkdir(parents=True, exist_ok=True)
archives = list(DOWNLOAD.glob('*.zip'))
if not archives:
    subprocess.run(['kaggle', 'datasets', 'download', '-d',
                    'ahmedelbanby/faceforensicsplusplus-c23-deepfakebench-structure',
                    '-p', str(DOWNLOAD)], check=True)
    archives = list(DOWNLOAD.glob('*.zip'))
if len(archives) != 1:
    raise RuntimeError(f'Expected one Kaggle ZIP, found {[p.name for p in archives]}')
ARCHIVE = archives[0]
print('Downloaded archive to temporary Colab storage:', ARCHIVE.name, f'{ARCHIVE.stat().st_size:,} bytes')
print('Frozen checkpoint root:', CV_ROOT)


In [ ]:
# Extract only the FaceForensics++ subtree; skip unrelated datasets in the mixed collection.
WORK = Path('/content/kaggle_ffpp_c23_work')
EXTRACT_ROOT = WORK / 'extracted'
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ARCHIVE) as zf:
    members = [i for i in zf.infolist() if '/FaceForensics++/' in ('/' + i.filename.replace(chr(92), '/'))]
    if not members:
        raise RuntimeError('No FaceForensics++ subtree found in the Kaggle ZIP; stop and inspect the archive listing before evaluation.')
    needed = sum(i.file_size for i in members)
    free = shutil.disk_usage('/content').free
    if free < needed + 2 * 1024**3:
        raise RuntimeError(f'Insufficient temporary disk for FF++ extraction: need about {needed + 2 * 1024**3:,} bytes, have {free:,}')
    base = EXTRACT_ROOT.resolve()
    for item in members:
        target = (EXTRACT_ROOT / item.filename).resolve()
        if target != base and base not in target.parents:
            raise RuntimeError(f'Unsafe archive path: {item.filename}')
        if item.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(item) as src, target.open('wb') as dst:
                shutil.copyfileobj(src, dst)

roots = [p.parent for p in EXTRACT_ROOT.rglob('original_sequences')
         if p.is_dir() and (p.parent / 'manipulated_sequences').is_dir()]
if len(roots) != 1:
    raise RuntimeError(f'Expected one FF++ root after extraction, found {roots[:10]}')
FFPP_ROOT = roots[0]
print('FF++ subtree extracted:', FFPP_ROOT)
print('Files extracted:', len(members), '| extracted bytes:', needed)


In [ ]:
# Reuse the existing project model definition and install the FF++ video-level evaluator.
SOURCE_ZIP = WORK / 'poor_guys_deepfake_colab_source.zip'
SOURCE_ZIP.write_bytes(base64.b64decode(SOURCE_B64.read_bytes().strip()))
PROJECT_ROOT = WORK / 'project_source'
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(SOURCE_ZIP) as zf:
    zf.extractall(PROJECT_ROOT)
PROJECT = PROJECT_ROOT / 'poor_guys_deepfake_project'
sys.path.insert(0, str(PROJECT / 'multimodal'))
EVALUATOR = PROJECT / 'multimodal' / 'evaluate_ffpp_c23.py'
EVALUATOR.write_bytes(base64.b64decode('IiIiRnJvemVuIHZpc3VhbC1vbmx5IGV2YWx1YXRpb24gb2YgU0RGVkQyIENWIGNoZWNrcG9pbnRzIG9uIEZGKysgQzIzIGZyYW1lcy4iIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBjc3YKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKaW1wb3J0IHN0YXRpc3RpY3MKaW1wb3J0IHN5cwpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBkZWZhdWx0ZGljdApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCB0b3JjaApmcm9tIFBJTCBpbXBvcnQgSW1hZ2UKZnJvbSB0b3JjaC51dGlscy5kYXRhIGltcG9ydCBEYXRhTG9hZGVyLCBEYXRhc2V0CmZyb20gdG9yY2h2aXNpb24gaW1wb3J0IHRyYW5zZm9ybXMKCmZyb20gbW9kZWwgaW1wb3J0IGJ1aWxkX21vZGVsCgpJTUFHRV9FWFRFTlNJT05TID0geyIuanBnIiwgIi5qcGVnIiwgIi5wbmciLCAiLndlYnAiLCAiLmJtcCJ9CgoKZGVmIGF1YyhzY29yZXM6IGxpc3RbZmxvYXRdLCBsYWJlbHM6IGxpc3RbaW50XSkgLT4gZmxvYXQgfCBOb25lOgogICAgcG9zaXRpdmVzID0gc3VtKGxhYmVscykKICAgIG5lZ2F0aXZlcyA9IGxlbihsYWJlbHMpIC0gcG9zaXRpdmVzCiAgICBpZiBub3QgcG9zaXRpdmVzIG9yIG5vdCBuZWdhdGl2ZXM6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIG9yZGVyZWQgPSBzb3J0ZWQoemlwKHNjb3JlcywgbGFiZWxzKSkKICAgIHJhbmtfc3VtID0gMC4wCiAgICBpID0gMAogICAgd2hpbGUgaSA8IGxlbihvcmRlcmVkKToKICAgICAgICBqID0gaSArIDEKICAgICAgICB3aGlsZSBqIDwgbGVuKG9yZGVyZWQpIGFuZCBvcmRlcmVkW2pdWzBdID09IG9yZGVyZWRbaV1bMF06CiAgICAgICAgICAgIGogKz0gMQogICAgICAgIHJhbmtfc3VtICs9IHN1bShsYWJlbCBmb3IgXywgbGFiZWwgaW4gb3JkZXJlZFtpOmpdKSAqICgoaSArIDEgKyBqKSAvIDIpCiAgICAgICAgaSA9IGoKICAgIHJldHVybiAocmFua19zdW0gLSBwb3NpdGl2ZXMgKiAocG9zaXRpdmVzICsgMSkgLyAyKSAvIChwb3NpdGl2ZXMgKiBuZWdhdGl2ZXMpCgoKZGVmIG1ldHJpY3Moc2NvcmVzOiBsaXN0W2Zsb2F0XSwgbGFiZWxzOiBsaXN0W2ludF0sIHRocmVzaG9sZDogZmxvYXQpIC0+IGRpY3Q6CiAgICBwcmVkaWN0ZWQgPSBbaW50KHNjb3JlID49IHRocmVzaG9sZCkgZm9yIHNjb3JlIGluIHNjb3Jlc10KICAgIHRwID0gc3VtKHAgPT0gMSBhbmQgeSA9PSAxIGZvciBwLCB5IGluIHppcChwcmVkaWN0ZWQsIGxhYmVscykpCiAgICB0biA9IHN1bShwID09IDAgYW5kIHkgPT0gMCBmb3IgcCwgeSBpbiB6aXAocHJlZGljdGVkLCBsYWJlbHMpKQogICAgZnAgPSBzdW0ocCA9PSAxIGFuZCB5ID09IDAgZm9yIHAsIHkgaW4gemlwKHByZWRpY3RlZCwgbGFiZWxzKSkKICAgIGZuID0gc3VtKHAgPT0gMCBhbmQgeSA9PSAxIGZvciBwLCB5IGluIHppcChwcmVkaWN0ZWQsIGxhYmVscykpCiAgICBwb3MsIG5lZyA9IHN1bShsYWJlbHMpLCBsZW4obGFiZWxzKSAtIHN1bShsYWJlbHMpCiAgICB0cHIgPSB0cCAvIHBvcyBpZiBwb3MgZWxzZSBOb25lCiAgICB0bnIgPSB0biAvIG5lZyBpZiBuZWcgZWxzZSBOb25lCiAgICByZXR1cm4gewogICAgICAgICJ2aWRlb19jb3VudCI6IGxlbihsYWJlbHMpLAogICAgICAgICJjbGFzc19jb3VudHMiOiB7InJlYWwiOiBuZWcsICJzeW50aGV0aWMiOiBwb3N9LAogICAgICAgICJhY2N1cmFjeSI6ICh0cCArIHRuKSAvIGxlbihsYWJlbHMpIGlmIGxhYmVscyBlbHNlIE5vbmUsCiAgICAgICAgImJhbGFuY2VkX2FjY3VyYWN5IjogKHRwciArIHRucikgLyAyIGlmIHRwciBpcyBub3QgTm9uZSBhbmQgdG5yIGlzIG5vdCBOb25lIGVsc2UgTm9uZSwKICAgICAgICAicm9jX2F1YyI6IGF1YyhzY29yZXMsIGxhYmVscyksCiAgICAgICAgInRocmVzaG9sZF9mcm9tX3NkZnZkMl92YWxpZGF0aW9uIjogdGhyZXNob2xkLAogICAgICAgICJjb25mdXNpb25fbWF0cml4IjogeyJ0cnVlX3N5bnRoZXRpYyI6IHRwLCAiZmFsc2Vfc3ludGhldGljIjogZnAsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInRydWVfcmVhbCI6IHRuLCAibWlzc2VkX3N5bnRoZXRpYyI6IGZufSwKICAgIH0KCgpkZWYgbG9jYXRlX2ZmcHBfcm9vdChyb290OiBQYXRoKSAtPiBQYXRoOgogICAgaWYgKHJvb3QgLyAib3JpZ2luYWxfc2VxdWVuY2VzIikuaXNfZGlyKCkgYW5kIChyb290IC8gIm1hbmlwdWxhdGVkX3NlcXVlbmNlcyIpLmlzX2RpcigpOgogICAgICAgIHJldHVybiByb290CiAgICBjYW5kaWRhdGVzID0gW3AgZm9yIHAgaW4gcm9vdC5yZ2xvYigib3JpZ2luYWxfc2VxdWVuY2VzIikKICAgICAgICAgICAgICAgICAgaWYgcC5pc19kaXIoKSBhbmQgKHAucGFyZW50IC8gIm1hbmlwdWxhdGVkX3NlcXVlbmNlcyIpLmlzX2RpcigpXQogICAgaWYgbGVuKGNhbmRpZGF0ZXMpICE9IDE6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkV4cGVjdGVkIG9uZSBGRisrIHJvb3Qgd2l0aCBvcmlnaW5hbF9zZXF1ZW5jZXMgYW5kIG1hbmlwdWxhdGVkX3NlcXVlbmNlczsgZm91bmQge2xlbihjYW5kaWRhdGVzKX0iKQogICAgcmV0dXJuIGNhbmRpZGF0ZXNbMF0ucGFyZW50CgoKZGVmIHNlcXVlbmNlX2ZyYW1lcyhmb2xkZXI6IFBhdGgpIC0+IGxpc3RbUGF0aF06CiAgICByZXR1cm4gc29ydGVkKChwIGZvciBwIGluIGZvbGRlci5pdGVyZGlyKCkgaWYgcC5pc19maWxlKCkgYW5kIHAuc3VmZml4LmNhc2Vmb2xkKCkgaW4gSU1BR0VfRVhURU5TSU9OUyksCiAgICAgICAgICAgICAgICAgIGtleT1sYW1iZGEgcDogKDAsIGludChwLnN0ZW0pKSBpZiBwLnN0ZW0uaXNkaWdpdCgpIGVsc2UgKDEsIHAubmFtZS5jYXNlZm9sZCgpKSkKCgpkZWYgc2FtcGxlX2V2ZW4ocGF0aHM6IGxpc3RbUGF0aF0sIGNvdW50OiBpbnQpIC0+IGxpc3RbUGF0aF06CiAgICBpZiBsZW4ocGF0aHMpIDw9IGNvdW50OgogICAgICAgIHJldHVybiBwYXRocwogICAgaWYgY291bnQgPT0gMToKICAgICAgICByZXR1cm4gW3BhdGhzWyhsZW4ocGF0aHMpIC0gMSkgLy8gMl1dCiAgICBpbmRpY2VzID0gc29ydGVkKHtyb3VuZChpICogKGxlbihwYXRocykgLSAxKSAvIChjb3VudCAtIDEpKSBmb3IgaSBpbiByYW5nZShjb3VudCl9KQogICAgcmV0dXJuIFtwYXRoc1tpXSBmb3IgaSBpbiBpbmRpY2VzXQoKCmRlZiBkaXNjb3Zlcl9zZXF1ZW5jZXMocm9vdDogUGF0aCwgZnJhbWVzX3Blcl92aWRlbzogaW50KSAtPiB0dXBsZVtQYXRoLCBsaXN0W2RpY3RdLCBkaWN0XToKICAgIGZmcHAgPSBsb2NhdGVfZmZwcF9yb290KHJvb3QuZXhwYW5kdXNlcigpLnJlc29sdmUoKSkKICAgIHJlYWxfZnJhbWVfcm9vdHMgPSBsaXN0KGZmcHAuZ2xvYigib3JpZ2luYWxfc2VxdWVuY2VzLyoqL2MyMy9mcmFtZXMiKSkKICAgIGZha2VfZnJhbWVfcm9vdHMgPSBsaXN0KGZmcHAuZ2xvYigibWFuaXB1bGF0ZWRfc2VxdWVuY2VzLyovYzIzL2ZyYW1lcyIpKQogICAgaWYgbm90IHJlYWxfZnJhbWVfcm9vdHMgb3Igbm90IGZha2VfZnJhbWVfcm9vdHM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRkYrKyBDMjMgZnJhbWUgZm9sZGVycyBtaXNzaW5nOyBleHBlY3RlZCBvcmlnaW5hbF9zZXF1ZW5jZXMvKiovYzIzL2ZyYW1lcyBhbmQgbWFuaXB1bGF0ZWRfc2VxdWVuY2VzLyovYzIzL2ZyYW1lcyIpCiAgICBzZXF1ZW5jZXMsIHNraXBwZWQgPSBbXSwgW10KICAgIGZvciBmcmFtZV9yb290LCBsYWJlbCwgbWV0aG9kIGluIFsocCwgMCwgInJlYWwiKSBmb3IgcCBpbiByZWFsX2ZyYW1lX3Jvb3RzXSArIFsKICAgICAgICAocCwgMSwgcC5wYXJlbnRzWzFdLm5hbWUpIGZvciBwIGluIGZha2VfZnJhbWVfcm9vdHMKICAgIF06CiAgICAgICAgZm9yIGZvbGRlciBpbiBzb3J0ZWQocCBmb3IgcCBpbiBmcmFtZV9yb290Lml0ZXJkaXIoKSBpZiBwLmlzX2RpcigpKToKICAgICAgICAgICAgZnJhbWVzID0gc2VxdWVuY2VfZnJhbWVzKGZvbGRlcikKICAgICAgICAgICAgaWYgbm90IGZyYW1lczoKICAgICAgICAgICAgICAgIHNraXBwZWQuYXBwZW5kKGZvbGRlci5yZWxhdGl2ZV90byhmZnBwKS5hc19wb3NpeCgpKQogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgc2VxdWVuY2VzLmFwcGVuZCh7InNlcXVlbmNlIjogZm9sZGVyLnJlbGF0aXZlX3RvKGZmcHApLmFzX3Bvc2l4KCksICJsYWJlbCI6IGxhYmVsLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAibWV0aG9kIjogbWV0aG9kLCAiZnJhbWVfcGF0aHMiOiBzYW1wbGVfZXZlbihmcmFtZXMsIGZyYW1lc19wZXJfdmlkZW8pLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAiYXZhaWxhYmxlX2ZyYW1lX2NvdW50IjogbGVuKGZyYW1lcyl9KQogICAgc2VlbiA9IHNldCgpCiAgICB1bmlxdWUgPSBbXQogICAgZm9yIHJvdyBpbiBzb3J0ZWQoc2VxdWVuY2VzLCBrZXk9bGFtYmRhIHI6IChyWyJsYWJlbCJdLCByWyJtZXRob2QiXS5jYXNlZm9sZCgpLCByWyJzZXF1ZW5jZSJdKSk6CiAgICAgICAga2V5ID0gcm93WyJzZXF1ZW5jZSJdCiAgICAgICAgaWYga2V5IG5vdCBpbiBzZWVuOgogICAgICAgICAgICBzZWVuLmFkZChrZXkpCiAgICAgICAgICAgIHVuaXF1ZS5hcHBlbmQocm93KQogICAgbGFiZWxzID0gW3JbImxhYmVsIl0gZm9yIHIgaW4gdW5pcXVlXQogICAgaWYgbm90IHVuaXF1ZSBvciBub3QgYW55KGxhYmVscykgb3IgYWxsKGxhYmVscyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRkYrKyBpbnZlbnRvcnkgbXVzdCBpbmNsdWRlIGF0IGxlYXN0IG9uZSByZWFsIGFuZCBvbmUgbWFuaXB1bGF0ZWQgc2VxdWVuY2UiKQogICAgYXVkaXQgPSB7InJlYWxfc2VxdWVuY2VzIjogc3VtKHJbImxhYmVsIl0gPT0gMCBmb3IgciBpbiB1bmlxdWUpLAogICAgICAgICAgICAgIm1hbmlwdWxhdGVkX3NlcXVlbmNlcyI6IHN1bShyWyJsYWJlbCJdID09IDEgZm9yIHIgaW4gdW5pcXVlKSwKICAgICAgICAgICAgICJzZXF1ZW5jZXNfYnlfbWV0aG9kIjoge30sICJzZXF1ZW5jZXNfd2l0aG91dF9mcmFtZXMiOiBsZW4oc2tpcHBlZCksCiAgICAgICAgICAgICAic2tpcHBlZF9leGFtcGxlcyI6IHNraXBwZWRbOjIwXSwgInNhbXBsZWRfZnJhbWVzX3Blcl92aWRlb19tYXgiOiBmcmFtZXNfcGVyX3ZpZGVvfQogICAgZm9yIG1ldGhvZCBpbiBzb3J0ZWQoe3JbIm1ldGhvZCJdIGZvciByIGluIHVuaXF1ZX0sIGtleT1zdHIuY2FzZWZvbGQpOgogICAgICAgIGF1ZGl0WyJzZXF1ZW5jZXNfYnlfbWV0aG9kIl1bbWV0aG9kXSA9IHN1bShyWyJtZXRob2QiXSA9PSBtZXRob2QgZm9yIHIgaW4gdW5pcXVlKQogICAgcmV0dXJuIGZmcHAsIHVuaXF1ZSwgYXVkaXQKCgpjbGFzcyBGcmFtZXMoRGF0YXNldCk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgaXRlbXM6IGxpc3RbdHVwbGVbaW50LCBQYXRoXV0sIHNpemU6IGludCk6CiAgICAgICAgc2VsZi5pdGVtcyA9IGl0ZW1zCiAgICAgICAgc2VsZi50cmFuc2Zvcm0gPSB0cmFuc2Zvcm1zLkNvbXBvc2UoWwogICAgICAgICAgICB0cmFuc2Zvcm1zLlJlc2l6ZSgoc2l6ZSwgc2l6ZSkpLCB0cmFuc2Zvcm1zLlRvVGVuc29yKCksCiAgICAgICAgICAgIHRyYW5zZm9ybXMuTm9ybWFsaXplKCguNDg1LCAuNDU2LCAuNDA2KSwgKC4yMjksIC4yMjQsIC4yMjUpKSwKICAgICAgICBdKQoKICAgIGRlZiBfX2xlbl9fKHNlbGYpOgogICAgICAgIHJldHVybiBsZW4oc2VsZi5pdGVtcykKCiAgICBkZWYgX19nZXRpdGVtX18oc2VsZiwgaSk6CiAgICAgICAgc2VxdWVuY2VfaW5kZXgsIHBhdGggPSBzZWxmLml0ZW1zW2ldCiAgICAgICAgd2l0aCBJbWFnZS5vcGVuKHBhdGgpIGFzIGltYWdlOgogICAgICAgICAgICB0ZW5zb3IgPSBzZWxmLnRyYW5zZm9ybShpbWFnZS5jb252ZXJ0KCJSR0IiKSkKICAgICAgICByZXR1cm4gdGVuc29yLCBzZXF1ZW5jZV9pbmRleAoKCmRlZiBmb2xkcyhjdl9yb290OiBQYXRoKSAtPiBsaXN0W3R1cGxlW1BhdGgsIFBhdGgsIFBhdGhdXToKICAgIHJlc3VsdCA9IFtdCiAgICBmb3IgZm9sZF9kaXIgaW4gc29ydGVkKGN2X3Jvb3QuZ2xvYigiZm9sZF8qIikpOgogICAgICAgIGNoZWNrcG9pbnQgPSBmb2xkX2RpciAvICJtdWx0aW1vZGFsIiAvICJiZXN0X2F2X21vZGVsLnB0IgogICAgICAgIHRocmVzaG9sZCA9IGZvbGRfZGlyIC8gInRocmVzaG9sZF92aXN1YWwuanNvbiIKICAgICAgICBpZiBjaGVja3BvaW50LmlzX2ZpbGUoKSBhbmQgdGhyZXNob2xkLmlzX2ZpbGUoKToKICAgICAgICAgICAgcmVzdWx0LmFwcGVuZCgoZm9sZF9kaXIsIGNoZWNrcG9pbnQsIHRocmVzaG9sZCkpCiAgICBpZiBsZW4ocmVzdWx0KSAhPSA1OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJFeHBlY3RlZCBleGFjdGx5IGZpdmUgY29tcGxldGUgU0RGVkQyIGZvbGQgY2hlY2twb2ludHMgdW5kZXIge2N2X3Jvb3R9LCBmb3VuZCB7bGVuKHJlc3VsdCl9IikKICAgIHJldHVybiByZXN1bHQKCgpkZWYgbWFpbigpIC0+IGludDoKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18pCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWZyYW1lcy1yb290IiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlLCBoZWxwPSJGRisrIEMyMyByb290IG9yIGV4dHJhY3RlZCBjb2xsZWN0aW9uIHJvb3QiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1jdi1yb290IiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlLCBoZWxwPSJTREZWRDIgcGFpcmVkLXNvdXJjZSBncm91cF9jdiBkaXJlY3RvcnkiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdXRwdXQtZGlyIiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1mcmFtZXMtcGVyLXZpZGVvIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tYmF0Y2gtc2l6ZSIsIHR5cGU9aW50LCBkZWZhdWx0PTY0KQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS13b3JrZXJzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZGV2aWNlIiwgY2hvaWNlcz0oImF1dG8iLCAiY3B1IiwgImN1ZGEiLCAibXBzIiksIGRlZmF1bHQ9ImF1dG8iKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKICAgIHRyeToKICAgICAgICBjdl9yb290ID0gYXJncy5jdl9yb290LmV4cGFuZHVzZXIoKS5yZXNvbHZlKCkKICAgICAgICBvdXRwdXQgPSBhcmdzLm91dHB1dF9kaXIuZXhwYW5kdXNlcigpLnJlc29sdmUoKQogICAgICAgIGlmIG5vdCBjdl9yb290LmlzX2RpcigpIG9yIGFyZ3MuZnJhbWVzX3Blcl92aWRlbyA8IDEgb3IgYXJncy5iYXRjaF9zaXplIDwgMSBvciBhcmdzLndvcmtlcnMgPCAwOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJJbnZhbGlkIGNoZWNrcG9pbnQvZGF0YSBwYXRocyBvciBub25wb3NpdGl2ZSBmcmFtZS9iYXRjaCBzZXR0aW5ncyIpCiAgICAgICAgaWYgb3V0cHV0LmV4aXN0cygpIGFuZCAobm90IG91dHB1dC5pc19kaXIoKSBvciBhbnkob3V0cHV0Lml0ZXJkaXIoKSkpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiT3V0cHV0IGRpcmVjdG9yeSBtdXN0IGJlIG5ldyBvciBlbXB0eToge291dHB1dH0iKQogICAgICAgIGZmcHBfcm9vdCwgc2VxdWVuY2VzLCBhdWRpdCA9IGRpc2NvdmVyX3NlcXVlbmNlcyhhcmdzLmZyYW1lc19yb290LCBhcmdzLmZyYW1lc19wZXJfdmlkZW8pCiAgICAgICAgZGV2aWNlX25hbWUgPSBhcmdzLmRldmljZQogICAgICAgIGlmIGRldmljZV9uYW1lID09ICJhdXRvIjoKICAgICAgICAgICAgZGV2aWNlX25hbWUgPSAiY3VkYSIgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICJtcHMiIGlmIGhhc2F0dHIodG9yY2guYmFja2VuZHMsICJtcHMiKSBhbmQgdG9yY2guYmFja2VuZHMubXBzLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIKICAgICAgICBpZiBkZXZpY2VfbmFtZSA9PSAiY3VkYSIgYW5kIG5vdCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJDVURBIHJlcXVlc3RlZCBidXQgdW5hdmFpbGFibGUiKQogICAgICAgIGlmIGRldmljZV9uYW1lID09ICJtcHMiIGFuZCBub3QgKGhhc2F0dHIodG9yY2guYmFja2VuZHMsICJtcHMiKSBhbmQgdG9yY2guYmFja2VuZHMubXBzLmlzX2F2YWlsYWJsZSgpKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiTVBTIHJlcXVlc3RlZCBidXQgdW5hdmFpbGFibGUiKQogICAgICAgIGRldmljZSA9IHRvcmNoLmRldmljZShkZXZpY2VfbmFtZSkKICAgICAgICBvdXRwdXQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgIGZyYW1lX2l0ZW1zID0gWyhzaSwgZnJhbWUpIGZvciBzaSwgc2VxIGluIGVudW1lcmF0ZShzZXF1ZW5jZXMpIGZvciBmcmFtZSBpbiBzZXFbImZyYW1lX3BhdGhzIl1dCiAgICAgICAgZm9sZF9yZXBvcnRzLCBwcmVkaWN0aW9ucyA9IFtdLCBbXQogICAgICAgIGZvciBmb2xkX2RpciwgY2hlY2twb2ludF9wYXRoLCB0aHJlc2hvbGRfcGF0aCBpbiBmb2xkcyhjdl9yb290KToKICAgICAgICAgICAgdGhyZXNob2xkID0gZmxvYXQoanNvbi5sb2Fkcyh0aHJlc2hvbGRfcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpWyJzZWxlY3RlZF90aHJlc2hvbGQiXSkKICAgICAgICAgICAgaWYgbm90IG1hdGguaXNmaW5pdGUodGhyZXNob2xkKSBvciBub3QgMCA8PSB0aHJlc2hvbGQgPD0gMToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJJbnZhbGlkIHZpc3VhbCB0aHJlc2hvbGQ6IHt0aHJlc2hvbGRfcGF0aH0iKQogICAgICAgICAgICBja3B0ID0gdG9yY2gubG9hZChjaGVja3BvaW50X3BhdGgsIG1hcF9sb2NhdGlvbj0iY3B1Iiwgd2VpZ2h0c19vbmx5PUZhbHNlKQogICAgICAgICAgICBtb2RlbCA9IGJ1aWxkX21vZGVsKHByZXRyYWluZWRfdmlzdWFsPUZhbHNlKS50byhkZXZpY2UpCiAgICAgICAgICAgIG1vZGVsLmxvYWRfc3RhdGVfZGljdChja3B0WyJzdGF0ZV9kaWN0Il0pCiAgICAgICAgICAgIG1vZGVsLmV2YWwoKQogICAgICAgICAgICBsb2FkZXIgPSBEYXRhTG9hZGVyKEZyYW1lcyhmcmFtZV9pdGVtcywgaW50KGNrcHQuZ2V0KCJpbWFnZV9zaXplIiwgMjI0KSkpLCBiYXRjaF9zaXplPWFyZ3MuYmF0Y2hfc2l6ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzaHVmZmxlPUZhbHNlLCBudW1fd29ya2Vycz1hcmdzLndvcmtlcnMsIHBpbl9tZW1vcnk9ZGV2aWNlLnR5cGUgPT0gImN1ZGEiKQogICAgICAgICAgICBwZXJfc2VxdWVuY2UgPSBbW10gZm9yIF8gaW4gc2VxdWVuY2VzXQogICAgICAgICAgICB3aXRoIHRvcmNoLmluZmVyZW5jZV9tb2RlKCk6CiAgICAgICAgICAgICAgICBmb3IgaW1hZ2VzLCBzZXFfaWRzIGluIGxvYWRlcjoKICAgICAgICAgICAgICAgICAgICBsb2dpdHMgPSBtb2RlbC52aXN1YWxfaGVhZChtb2RlbC52aXN1YWwoaW1hZ2VzLnRvKGRldmljZSkpKS5zcXVlZXplKDEpCiAgICAgICAgICAgICAgICAgICAgc2NvcmVzID0gdG9yY2guc2lnbW9pZChsb2dpdHMpLmNwdSgpLnRvbGlzdCgpCiAgICAgICAgICAgICAgICAgICAgZm9yIHNpLCBzY29yZSBpbiB6aXAoc2VxX2lkcy50b2xpc3QoKSwgc2NvcmVzKToKICAgICAgICAgICAgICAgICAgICAgICAgcGVyX3NlcXVlbmNlW3NpXS5hcHBlbmQoZmxvYXQoc2NvcmUpKQogICAgICAgICAgICBpZiBhbnkobm90IHZhbHVlcyBmb3IgdmFsdWVzIGluIHBlcl9zZXF1ZW5jZSk6CiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJTb21lIHZpZGVvcyByZWNlaXZlZCBubyBzY29yZXMgaW4ge2ZvbGRfZGlyLm5hbWV9IikKICAgICAgICAgICAgdmlkZW9fc2NvcmVzID0gW3N0YXRpc3RpY3MubWVhbih2YWx1ZXMpIGZvciB2YWx1ZXMgaW4gcGVyX3NlcXVlbmNlXQogICAgICAgICAgICBsYWJlbHMgPSBbclsibGFiZWwiXSBmb3IgciBpbiBzZXF1ZW5jZXNdCiAgICAgICAgICAgIGZvbGRfbWV0cmljcyA9IHsiZm9sZCI6IGZvbGRfZGlyLm5hbWUsICoqbWV0cmljcyh2aWRlb19zY29yZXMsIGxhYmVscywgdGhyZXNob2xkKX0KICAgICAgICAgICAgZm9sZF9yZXBvcnRzLmFwcGVuZChmb2xkX21ldHJpY3MpCiAgICAgICAgICAgIGZvciBzZXEsIHNjb3JlLCBmcmFtZV9zY29yZXMgaW4gemlwKHNlcXVlbmNlcywgdmlkZW9fc2NvcmVzLCBwZXJfc2VxdWVuY2UpOgogICAgICAgICAgICAgICAgcHJlZGljdGlvbnMuYXBwZW5kKHsiZm9sZCI6IGZvbGRfZGlyLm5hbWUsICJzZXF1ZW5jZSI6IHNlcVsic2VxdWVuY2UiXSwgIm1ldGhvZCI6IHNlcVsibWV0aG9kIl0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJsYWJlbCI6ICJzeW50aGV0aWMiIGlmIHNlcVsibGFiZWwiXSBlbHNlICJyZWFsIiwgInNhbXBsZWRfZnJhbWVfY291bnQiOiBsZW4oZnJhbWVfc2NvcmVzKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgImF2YWlsYWJsZV9mcmFtZV9jb3VudCI6IHNlcVsiYXZhaWxhYmxlX2ZyYW1lX2NvdW50Il0sICJzY29yZV9zeW50aGV0aWMiOiBzY29yZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInRocmVzaG9sZF9mcm9tX3NkZnZkMl92YWxpZGF0aW9uIjogdGhyZXNob2xkLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAicHJlZGljdGlvbiI6ICJzeW50aGV0aWMiIGlmIHNjb3JlID49IHRocmVzaG9sZCBlbHNlICJyZWFsIn0pCiAgICAgICAgICAgIHByaW50KGYiU2NvcmVkIHtmb2xkX2Rpci5uYW1lfToge2xlbihzZXF1ZW5jZXMpfSB2aWRlb3MiLCBmbHVzaD1UcnVlKQogICAgICAgICAgICBkZWwgbW9kZWwKICAgICAgICAgICAgaWYgZGV2aWNlLnR5cGUgPT0gImN1ZGEiOgogICAgICAgICAgICAgICAgdG9yY2guY3VkYS5lbXB0eV9jYWNoZSgpCiAgICAgICAgZm9sZF9tZWFuID0ge30KICAgICAgICBmb3Iga2V5IGluICgiYWNjdXJhY3kiLCAiYmFsYW5jZWRfYWNjdXJhY3kiLCAicm9jX2F1YyIpOgogICAgICAgICAgICB2YWxzID0gW3Jba2V5XSBmb3IgciBpbiBmb2xkX3JlcG9ydHMgaWYgcltrZXldIGlzIG5vdCBOb25lXQogICAgICAgICAgICBmb2xkX21lYW5ba2V5XSA9IHsibWVhbiI6IHN0YXRpc3RpY3MubWVhbih2YWxzKSwgInN0ZGV2Ijogc3RhdGlzdGljcy5zdGRldih2YWxzKSBpZiBsZW4odmFscykgPiAxIGVsc2UgMC4wLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAiZm9sZF92YWx1ZXMiOiB2YWxzfQogICAgICAgIHJlcG9ydCA9IHsKICAgICAgICAgICAgInNjaGVtYV92ZXJzaW9uIjogIjEuMCIsICJkYXRhc2V0IjogIkthZ2dsZSBGRisrIEMyMyAoRmFjZUZvcmVuc2ljcysrIHN1YnRyZWUgb25seSkiLAogICAgICAgICAgICAiZXZhbHVhdGlvbiI6ICJmcm96ZW4gZXh0ZXJuYWwgdmlzdWFsIGV2YWx1YXRpb247IGFsbCBpbnZlbnRvcmllZCBGRisrIEMyMyBzZXF1ZW5jZXM7IG5vIHRyYWluaW5nIG9yIHRocmVzaG9sZCB0dW5pbmciLAogICAgICAgICAgICAidHJhaW5pbmdfcGVyZm9ybWVkIjogRmFsc2UsICJzY29yZV91bml0IjogInZpZGVvIHNlcXVlbmNlOyBzY29yZSBpcyBtZWFuIG9mIGRldGVybWluaXN0aWMgZXZlbmx5IHNhbXBsZWQgZnJhbWUgc2NvcmVzIiwKICAgICAgICAgICAgImZmcHBfcm9vdCI6IGZmcHBfcm9vdC5uYW1lLCAiaW52ZW50b3J5IjogYXVkaXQsICJmb2xkcyI6IGZvbGRfcmVwb3J0cywgImZvbGRfbWVhbiI6IGZvbGRfbWVhbiwKICAgICAgICAgICAgImxpbWl0YXRpb25zIjogWwogICAgICAgICAgICAgICAgIkthZ2dsZSB1cGxvYWQgaXMgYSBtaXhlZCBjb2xsZWN0aW9uOyBldmFsdWF0aW9uIG9ubHkgc2NhbnMgdGhlIEZhY2VGb3JlbnNpY3MrKyBzdWJ0cmVlIGFuZCBpZ25vcmVzIG90aGVyIGRhdGFzZXRzLiIsCiAgICAgICAgICAgICAgICAiS2FnZ2xlIGxhYmVscyBhbmQgZmlsZSBwcm92ZW5hbmNlIGFyZSBub3QgaW5kZXBlbmRlbnRseSBhdXRoZW50aWNhdGVkOyBjb21wYXJlIGludmVudG9yeSB3aXRoIG9mZmljaWFsIEZGKysgcmVjb3JkcyBiZWZvcmUgdHJlYXRpbmcgYXMgYmVuY2htYXJrLWdyYWRlLiIsCiAgICAgICAgICAgICAgICAiVGhyZXNob2xkcyBhcmUgZWFjaCBmb2xkJ3MgU0RGVkQyIHZpc3VhbCB2YWxpZGF0aW9uIHRocmVzaG9sZDsgbm8gdGhyZXNob2xkIHdhcyBmaXR0ZWQgb24gRkYrKy4iLAogICAgICAgICAgICAgICAgIkZyYW1lIHNjb3JlcyBhcmUgYXZlcmFnZWQgcGVyIHZpZGVvLCBidXQgcmVsYXRlZCBvcmlnaW5hbHMgYW5kIG1hbmlwdWxhdGlvbnMgbWF5IHNoYXJlIHNvdXJjZSBpZGVudGl0aWVzOyBzb3VyY2UgaW5kZXBlbmRlbmNlIGlzIHVudmVyaWZpZWQuIiwKICAgICAgICAgICAgICAgICJUaGlzIGlzIHZpZGVvLW9ubHkgdmlzdWFsIGV2YWx1YXRpb24gYW5kIHNheXMgbm90aGluZyBhYm91dCB0aGUgYXVkaW8gYnJhbmNoIG9yIGZ1c2VkIG11bHRpbW9kYWwgYWNjdXJhY3kuIiwKICAgICAgICAgICAgICAgICJVc2UgaXMgc3ViamVjdCB0byB0aGUgb3JpZ2luYWwgRmFjZUZvcmVuc2ljcyB0ZXJtcyBkZXNwaXRlIGFueSBLYWdnbGUtbGV2ZWwgbGlzdGluZyBsaWNlbnNlLiIKICAgICAgICAgICAgXQogICAgICAgIH0KICAgICAgICAob3V0cHV0IC8gImV4dGVybmFsX2V2YWx1YXRpb24uanNvbiIpLndyaXRlX3RleHQoanNvbi5kdW1wcyhyZXBvcnQsIGluZGVudD0yKSArICJcbiIsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICAgICAgd2l0aCAob3V0cHV0IC8gImV4dGVybmFsX3ByZWRpY3Rpb25zLmNzdiIpLm9wZW4oInciLCBuZXdsaW5lPSIiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBzdHJlYW06CiAgICAgICAgICAgIHdyaXRlciA9IGNzdi5EaWN0V3JpdGVyKHN0cmVhbSwgZmllbGRuYW1lcz1saXN0KHByZWRpY3Rpb25zWzBdKSkKICAgICAgICAgICAgd3JpdGVyLndyaXRlaGVhZGVyKCkKICAgICAgICAgICAgd3JpdGVyLndyaXRlcm93cyhwcmVkaWN0aW9ucykKICAgICAgICBwcmludChqc29uLmR1bXBzKHsiaW52ZW50b3J5IjogYXVkaXQsICJmb2xkX21lYW4iOiBmb2xkX21lYW4sICJkZXZpY2UiOiBzdHIoZGV2aWNlKX0sIGluZGVudD0yKSkKICAgICAgICBwcmludChmIlJlcG9ydDoge291dHB1dCAvICdleHRlcm5hbF9ldmFsdWF0aW9uLmpzb24nfSIpCiAgICAgICAgcHJpbnQoZiJQcmVkaWN0aW9uczoge291dHB1dCAvICdleHRlcm5hbF9wcmVkaWN0aW9ucy5jc3YnfSIpCiAgICAgICAgcmV0dXJuIDAKICAgIGV4Y2VwdCAoT1NFcnJvciwgVmFsdWVFcnJvciwgS2V5RXJyb3IsIGpzb24uSlNPTkRlY29kZUVycm9yLCBSdW50aW1lRXJyb3IpIGFzIGV4YzoKICAgICAgICBwcmludChmIkVycm9yOiB7ZXhjfSIsIGZpbGU9c3lzLnN0ZGVycikKICAgICAgICByZXR1cm4gMgoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICByYWlzZSBTeXN0ZW1FeGl0KG1haW4oKSkK'))
print('Evaluator ready:', EVALUATOR)


In [ ]:
# Run the five fold checkpoints; no fitting or threshold selection occurs on FF++.
OUTPUT = DEST_ROOT / f'external_eval_{time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())}'
cmd = [sys.executable, str(EVALUATOR), '--frames-root', str(FFPP_ROOT),
       '--cv-root', str(CV_ROOT), '--output-dir', str(OUTPUT),
       '--frames-per-video', '5', '--batch-size', '64', '--workers', '2', '--device', 'auto']
print('Starting frozen visual-only external evaluation...')
subprocess.run(cmd, cwd=PROJECT, check=True)
print('Drive report:', OUTPUT / 'external_evaluation.json')
print('Drive predictions:', OUTPUT / 'external_predictions.csv')


## Interpreting the result

This is an external visual-only result on the FF++ subtree. It does not validate audio or multimodal fusion. The package also contains other datasets, which this notebook deliberately ignores. Its Kaggle description and CC0 label do not replace the original FaceForensics terms; retain the original non-commercial research/education conditions. Treat the result as exploratory until the extracted inventory and method counts are compared with your approved FF++ release and its source-video lineage is confirmed.
